### 1. Raw 19F chemical-shift distribution

Purpose: Inspect the experimental shift distribution before any curation and export a publication-ready histogram.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# Plot the experimental 19F chemical-shift distribution
# Purpose: Load the raw shift dataset, validate the required columns, and export the distribution plot used as the starting overview of the workflow.
# -----------------------------------------------------------------------------

from pathlib import Path
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# ---------- Paths ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

PROJECT_ROOT = Path(".").resolve()
SHIFT_CSV_PATH = PROJECT_ROOT / "data" / "raw" / "shift_data.csv"
FIG_DIR = PROJECT_ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Base style ----------
apply_plot_style()

# Final manuscript-oriented rcParams
apply_plot_style(profile='compact')

# ---------- Load CSV ----------
if not SHIFT_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Shift CSV file not found:\n  {SHIFT_CSV_PATH}\n"
        "Please place the file at data/raw/shift_data.csv before running this cell."
    )

df = pd.read_csv(SHIFT_CSV_PATH)

# ---------- Validate & clean ----------
required_cols = {"SMILES", "shift_value"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        f"Found columns: {list(df.columns)}"
    )

df["SMILES"] = df["SMILES"].astype(str).str.strip()
df["shift_value"] = pd.to_numeric(df["shift_value"], errors="coerce")

n_before = len(df)
df = df.dropna(subset=["SMILES", "shift_value"]).reset_index(drop=True)
n_after = len(df)

print_section("Input dataset overview")
print_key_value_rows([
    ("Project root", PROJECT_ROOT),
    ("Input file", SHIFT_CSV_PATH),
    ("Rows before cleaning", n_before),
    ("Rows after cleaning", n_after),
])
display(prepare_display_table(df.head()))

# ---------- Data ----------
values = df["shift_value"].to_numpy()
n = len(values)

# ---------- Histogram bins ----------
# Freedman–Diaconis rule with conservative bounds for a clean publication figure
q25, q75 = np.percentile(values, [25, 75])
iqr = q75 - q25

if iqr > 0:
    bin_width = 2 * iqr * (n ** (-1/3))
    bins = math.ceil((values.max() - values.min()) / bin_width) if bin_width > 0 else 30
else:
    bins = int(np.sqrt(n))

bins = max(22, min(bins, 38))

print_section("Histogram settings")
print_key_value_rows([
    ("Final sample size (n)", n),
    ("Number of bins", bins),
    ("Minimum shift / ppm", f"{values.min():.2f}"),
    ("Maximum shift / ppm", f"{values.max():.2f}"),
])

# ---------- Plot ----------
# Use your existing figure template; if 'single' exists and is narrower than
# 'single_wide', prefer that. Otherwise keep 'single_wide'.
fig, ax = create_single_axis_figure("single_wide")

counts, _, _ = ax.hist(
    values,
    bins=bins,
    color="#D9D9D9",
    edgecolor="black",
    linewidth=0.55
)

# ---------- Labels ----------
ax.set_xlabel(r"$\delta_{\mathrm{exp}}$ (ppm)", labelpad=2)
ax.set_ylabel("Count", labelpad=2)

# ---------- Axes ----------
# Final version: keep all four spines for a more conservative journal look
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.7)

ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.tick_params(axis="both", which="major", pad=1.5)

# Tight y-limit with a little headroom
ax.set_ylim(0, max(counts) * 1.08)

# ---------- In-panel annotation ----------
ax.text(
    0.97, 0.96,
    f"n = {n}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=7
)

# ---------- Layout ----------
# No in-figure title; caption will carry the title in the manuscript
fig.tight_layout(pad=0.35)

# ---------- Save ----------
fig_path = FIG_DIR / "Figure_1_shift_distribution.png"
saved_paths = save_figure(fig, fig_path)

plt.show()
report_saved_paths(saved_paths, "Saved figure files")


### 2. Dataset cleaning and range filtering

Purpose: Remove duplicate or implausible entries while preserving the original curation logic used in the project.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 2. Dataset cleaning and range filtering
# Purpose: Remove duplicate or implausible entries while preserving the original curation logic used in the project.
# -----------------------------------------------------------------------------

from pathlib import Path

# ---------- Define output path ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CSV_PATH = PROCESSED_DIR / "cleaned_shift_data.csv"

# ---------- Initial statistics ----------
n_initial = len(df)

# ---------- Remove duplicated entries ----------
# Duplicates are defined as identical SMILES + identical shift_value
n_dup = df.duplicated(subset=["SMILES", "shift_value"]).sum()

df_cleaned = df.drop_duplicates(subset=["SMILES", "shift_value"]).copy()

# ---------- Remove extreme shift values ----------
# According to project decision:
# Remove shift_value > 0 ppm and shift_value < -250 ppm
condition = (df_cleaned["shift_value"] <= 0) & (df_cleaned["shift_value"] >= -250)

n_out_of_range = (~condition).sum()

df_cleaned = df_cleaned[condition].reset_index(drop=True)

# ---------- Final statistics ----------
n_final = len(df_cleaned)

print_section("Data Cleaning Summary")
print_key_value_rows([
    ('Initial sample count', n_initial),
    ('Duplicate entries removed', n_dup),
    ('Out-of-range entries removed', n_out_of_range),
    ('Final sample count', n_final),
])

# ---------- Save cleaned dataset ----------
df_cleaned.to_csv(CLEANED_CSV_PATH, index=False)

report_saved_paths([CLEANED_CSV_PATH], 'Saved cleaned datasets')
display(prepare_display_table(df_cleaned.head()))


### 3. Molecular feature construction without shielding

Purpose: Generate the descriptor and fingerprint matrix used for the structure-only machine-learning baseline.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 3. Molecular feature construction without shielding
# Purpose: Generate the descriptor and fingerprint matrix used for the structure-only machine-learning baseline.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import MACCSkeys, Descriptors
from rdkit import DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

# ---------- Input check ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

if "df_cleaned" not in globals():
    raise RuntimeError("df_cleaned not found. Please run Cell 2 (data cleaning) first.")

required_cols = {"SMILES", "shift_value"}
missing_cols = required_cols - set(df_cleaned.columns)
if missing_cols:
    raise ValueError(f"df_cleaned missing columns: {missing_cols}")

# ---------- Paths ----------
PROJECT_ROOT = Path(".").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = PROCESSED_DIR / "features_dataset.csv"
OUT_PARQUET = PROCESSED_DIR / "features_dataset.parquet"

# ---------- Parameters ----------
N_BITS = 1024
MORGAN_RADIUS = 2
ECFP_RADIUS = 4

# ---------- Fingerprint generators (RDKit-compatible signature) ----------
# NOTE: In your RDKit build, the argument name is includeChirality (not useChirality).
morgan_gen = GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=N_BITS, includeChirality=False)
ecfp_gen = GetMorganGenerator(radius=ECFP_RADIUS, fpSize=N_BITS, includeChirality=True)

# =========================
# Helper functions
# =========================

def mol_from_smiles(smiles: str):
    """Convert SMILES to RDKit Mol object. Returns None if parsing fails."""
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

def bitvect_to_numpy(fp) -> np.ndarray:
    """Convert RDKit ExplicitBitVect to a numpy array of 0/1."""
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def compute_basic_descriptors(mol: Chem.Mol) -> dict:
    """Compute basic physicochemical descriptors (numeric)."""
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
        "RingCount": Descriptors.RingCount(mol),
    }

def compute_morgan_fp(mol: Chem.Mol) -> np.ndarray:
    """Compute Morgan fingerprint (1024 bits) using MorganGenerator."""
    fp = morgan_gen.GetFingerprint(mol)
    return bitvect_to_numpy(fp)

def compute_ecfp_per_fluorine_or(mol: Chem.Mol) -> np.ndarray:
    """
    Compute fluorine-centered ECFP-like fingerprints (1024 bits).
    For multiple F atoms, combine fingerprints using bitwise OR (max).

    NOTE:
    This implementation uses MorganGenerator with fromAtoms selection.
    It does NOT use 'useFeatures=True' (Feature Morgan) because the API differs across RDKit builds.
    """
    fluorines = [a for a in mol.GetAtoms() if a.GetSymbol() == "F"]
    if not fluorines:
        return np.zeros((N_BITS,), dtype=np.int8)

    combined = np.zeros((N_BITS,), dtype=np.int8)
    for atom in fluorines:
        fp = ecfp_gen.GetFingerprint(mol, fromAtoms=[atom.GetIdx()])
        combined = np.maximum(combined, bitvect_to_numpy(fp))
    return combined

def compute_maccs_fp(mol: Chem.Mol) -> np.ndarray:
    """
    Compute MACCS keys.
    RDKit typically returns 167 bits (including bit 0). We keep the native length.
    """
    fp = MACCSkeys.GenMACCSKeys(mol)
    return bitvect_to_numpy(fp)

# =========================
# Feature construction
# =========================

feature_rows = []
invalid_smiles_count = 0

for _, row in df_cleaned.iterrows():
    smiles = str(row["SMILES"]).strip()
    shift = float(row["shift_value"])

    mol = mol_from_smiles(smiles)
    if mol is None:
        invalid_smiles_count += 1
        continue

    basic_desc = compute_basic_descriptors(mol)
    morgan_fp = compute_morgan_fp(mol)
    ecfp_f_fp = compute_ecfp_per_fluorine_or(mol)
    maccs_fp = compute_maccs_fp(mol)

    feature_dict = {
        "SMILES": smiles,
        "shift_value": shift,
        **basic_desc,
    }

    for i, bit in enumerate(morgan_fp):
        feature_dict[f"Morgan_{i}"] = int(bit)

    for i, bit in enumerate(ecfp_f_fp):
        feature_dict[f"ECFP_F_{i}"] = int(bit)

    for i, bit in enumerate(maccs_fp):
        feature_dict[f"MACCS_{i}"] = int(bit)

    feature_rows.append(feature_dict)

df_features = pd.DataFrame(feature_rows)

# =========================
# Save dataset (CSV required; Parquet optional)
# =========================

df_features.to_csv(OUT_CSV, index=False)

try:
    df_features.to_parquet(OUT_PARQUET, index=False)
    parquet_msg = f"Saved Parquet to: {OUT_PARQUET}"
except Exception as e:
    parquet_msg = f"Parquet not saved (optional). Reason: {type(e).__name__}: {e}"

# =========================
# Summary
# =========================

n_desc = 7
n_morgan = N_BITS
n_ecfp = N_BITS
n_maccs = len([c for c in df_features.columns if c.startswith("MACCS_")])

print_section("Feature Engineering Summary")
print(f"Total input samples (df_cleaned) : {len(df_cleaned)}")
print(f"Invalid SMILES removed           : {invalid_smiles_count}")
print(f"Final dataset size               : {len(df_features)}")

print("\n----- Feature dimensions -----")
print(f"PhysChem descriptors             : {n_desc}")
print(f"Morgan fingerprint               : {n_morgan}")
print(f"Fluorine-centered ECFP-like (OR) : {n_ecfp}")
print(f"MACCS keys (native length)       : {n_maccs}")
print(f"Total feature dimension          : {df_features.shape[1] - 2}")  # excluding SMILES & shift_value

print("\n----- Output files -----")
print(f"Saved CSV to                     : {OUT_CSV}")
print_section('Saved feature files')
print(parquet_msg)

df_features.head()


### 4. Baseline model benchmark without shielding

Purpose: Compare standard regression models on the structure-only feature set and summarize their test-set errors.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 4. Baseline model benchmark without shielding
# Purpose: Compare standard regression models on the structure-only feature set and summarize their test-set errors.
# -----------------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

# ---------- Paths ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

PROJECT_ROOT = Path('.').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
TABLE_DIR = RESULTS_DIR / 'tables'
FIG_DIR = RESULTS_DIR / 'figures'

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

OUT_TABLE = TABLE_DIR / 'model_benchmark_results.csv'
OUT_FIG = FIG_DIR / 'model_benchmark_mae.png'

apply_plot_style()

# ---------- Load features ----------
FEATURE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'features_dataset.parquet'

if not FEATURE_PATH.exists():
    raise FileNotFoundError('Feature dataset not found. Run feature engineering first.')

df_features = pd.read_parquet(FEATURE_PATH)

X = df_features.drop(columns=['SMILES', 'shift_value'])
y = df_features['shift_value']

# ---------- Train-test split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

print_section('Dataset Summary')
print(f'Total samples : {len(X)}')
print(f'Train samples : {len(X_train)}')
print(f'Test samples  : {len(X_test)}')
print(f'Feature dim   : {X.shape[1]}')

# ---------- Define models ----------
models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42),
    'ExtraTrees': ExtraTreesRegressor(random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'AdaBoost': AdaBoostRegressor(random_state=42),
    'SVR': SVR(),
    'KNN': KNeighborsRegressor(),
}

# ---------- Train and evaluate ----------
results = []

for name, model in models.items():
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        results.append({
            'Model': name,
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2,
        })

        print(metric_row(name, mae, rmse, r2))

    except Exception as e:
        print(f'{name} failed: {e}')

df_results = pd.DataFrame(results).sort_values('MAE')

# ---------- Save table ----------
df_results.to_csv(OUT_TABLE, index=False)

print('Results saved to:')
print(OUT_TABLE.resolve())

display(prepare_display_table(df_results))

# ---------- Plot MAE comparison ----------
fig, ax = create_single_axis_figure('single_wide')
ax.bar(df_results['Model'], df_results['MAE'], color='#4C78A8')
ax.tick_params(axis='x', rotation=90)
ax.set_ylabel('MAE (ppm)')
ax.set_title('Benchmark of baseline regression models')
style_axes(ax, grid_axis='y')
fig.tight_layout()

saved_paths = save_figure(fig, OUT_FIG)
plt.show()

print('Saved figure files:')
report_saved_paths(saved_paths, 'Saved output files')


### 5. Coarse hyperparameter search without shielding

Purpose: Run the initial train/validation model-selection step for the structure-only workflow.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 5. Coarse hyperparameter search without shielding
# Purpose: Run the initial train/validation model-selection step for the structure-only workflow.
# -----------------------------------------------------------------------------

from pathlib import Path
import os, json, random, csv
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

# ---------- Reproducibility ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
N_ITER = 100
CV_FOLDS = 5
SEARCH_NJOBS = 20

def set_global_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_global_seed(SEED)

# ---------- Paths ----------
PROJECT_ROOT = Path(".").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
TABLE_DIR = RESULTS_DIR / "tables"
MODEL_DIR = RESULTS_DIR / "models"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Distinct filenames for WITH-shield pipeline
OUT_SUMMARY = TABLE_DIR / "optimized_model_results_train_val_test.csv"
OUT_VAL_RANK = TABLE_DIR / "validation_ranking.csv"
OUT_TEST_FINAL = TABLE_DIR / "final_test_performance.csv"

OUT_CV_DT = TABLE_DIR / "cv_results_decisiontree.csv"
OUT_CV_ET = TABLE_DIR / "cv_results_extratrees.csv"

# RF will be generated as two files then auto-merged into OUT_CV_RF_MERGED
OUT_CV_RF_BOOT = TABLE_DIR / "cv_results_randomforest_bootstrap.csv"
OUT_CV_RF_NOBOOT = TABLE_DIR / "cv_results_randomforest_no_bootstrap.csv"
OUT_CV_RF_MERGED = TABLE_DIR / "cv_results_randomforest.csv"  # <- used by later Cell 13

# Save models
MODEL_PATH_DT = MODEL_DIR / "best_on_val_DecisionTree.pkl"
MODEL_PATH_RF = MODEL_DIR / "best_on_val_RandomForest.pkl"
MODEL_PATH_ET = MODEL_DIR / "best_on_val_ExtraTrees.pkl"
FINAL_MODEL_OUT = MODEL_DIR / "final_best_model.pkl"

# Also save best params (paper appendix / reproducibility)
BEST_PARAMS_JSON = TABLE_DIR / "best_params.json"

# ---------- Load dataset ----------
FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "features_dataset.parquet"
if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Feature dataset not found: {FEATURE_PATH}\n"
        "Please run the feature-engineering cell for shift_data first."
    )

df_features = pd.read_parquet(FEATURE_PATH)

required_cols = {"SMILES", "shift_value"}
missing = required_cols - set(df_features.columns)
if missing:
    raise ValueError(
        f"Missing required columns in features file: {missing}\n"
        "Make sure shielding_value is kept as a numeric feature."
    )

X = df_features.drop(columns=["SMILES", "shift_value"])
y = df_features["shift_value"].astype(float)

# ---------- Train / Validation / Test split ----------
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=(1 / 9), random_state=SEED
)

print_section("Dataset Summary (WITH shielding_value)")
print(f"Total samples : {len(X)}")
print(f"Train samples : {len(X_train)}")
print(f"Val samples   : {len(X_val)}")
print(f"Test samples  : {len(X_test)}")
print(f"Feature dim   : {X.shape[1]}")

# ---------- Parameter spaces (Coarse) ----------
param_spaces = {
    "DecisionTree": {
        "criterion": ["squared_error", "friedman_mse", "absolute_error"],
        "splitter": ["best", "random"],
        "max_depth": [None, 5, 10, 20, 30, 50, 80, 120],
        "min_samples_split": [2, 5, 10, 20, 50, 100],
        "min_samples_leaf": [1, 2, 4, 8, 16, 32],
        "max_features": [None, "sqrt", "log2", 0.2, 0.4, 0.6, 0.8],
        "min_impurity_decrease": [0.0, 1e-7, 1e-6, 1e-5, 1e-4],
        "ccp_alpha": [0.0, 1e-7, 1e-6, 1e-5, 1e-4],
    },
    "ExtraTrees": {
        "n_estimators": [200, 400, 800, 1200, 1600],
        "criterion": ["squared_error", "absolute_error", "friedman_mse"],
        "max_depth": [None, 10, 20, 40, 60, 80, 120],
        "min_samples_split": [2, 5, 10, 20, 50],
        "min_samples_leaf": [1, 2, 4, 8, 16],
        "max_features": ["sqrt", "log2", None, 0.2, 0.4, 0.6, 0.8],
        "min_impurity_decrease": [0.0, 1e-7, 1e-6, 1e-5],
        "ccp_alpha": [0.0, 1e-7, 1e-6, 1e-5],
    },
}

rf_common = {
    "n_estimators": [200, 400, 800, 1200, 1600],
    "criterion": ["squared_error", "absolute_error", "friedman_mse"],
    "max_depth": [None, 10, 20, 40, 60, 80, 120],
    "min_samples_split": [2, 5, 10, 20, 50],
    "min_samples_leaf": [1, 2, 4, 8, 16],
    "max_features": ["sqrt", "log2", None, 0.2, 0.4, 0.6, 0.8],
    "min_impurity_decrease": [0.0, 1e-7, 1e-6, 1e-5],
    "ccp_alpha": [0.0, 1e-7, 1e-6, 1e-5],
}

rf_space_bootstrap = {**rf_common, "bootstrap": [True], "max_samples": [0.5, 0.7, 0.9]}
rf_space_no_bootstrap = {**rf_common, "bootstrap": [False]}

# ---------- Models ----------
models = {
    "DecisionTree": DecisionTreeRegressor(random_state=SEED),
    "ExtraTrees": ExtraTreesRegressor(random_state=SEED, n_jobs=SEARCH_NJOBS),
    "RandomForest": RandomForestRegressor(random_state=SEED, n_jobs=SEARCH_NJOBS),
}

# ---------- Helpers ----------
def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

def save_cv_results(df_cv: pd.DataFrame, out_path: Path):
    """Write CV results robustly (params may contain commas)."""
    df_cv.to_csv(out_path, index=False, quoting=csv.QUOTE_MINIMAL, escapechar="\\")
    print(f"CV results saved to: {out_path.resolve()}")

def run_random_search(estimator, param_space, n_iter, seed, out_cv_path):
    """RandomizedSearchCV on TRAIN only (KFold CV), optimizing neg-MAE."""
    cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_space,
        n_iter=n_iter,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=SEARCH_NJOBS,
        random_state=seed,
        verbose=2,
        return_train_score=True
    )
    search.fit(X_train, y_train)
    save_cv_results(pd.DataFrame(search.cv_results_), out_cv_path)
    return search

def merge_rf_cv(boot_path: Path, noboot_path: Path, out_merged: Path):
    """Merge two RF CV results and recompute global ranking by mean_test_score (higher is better)."""
    df_b = pd.read_csv(boot_path, engine="python")
    df_n = pd.read_csv(noboot_path, engine="python")
    df_b["rf_branch"] = "bootstrap_true"
    df_n["rf_branch"] = "bootstrap_false"

    cols = sorted(set(df_b.columns).union(set(df_n.columns)))
    df_b = df_b.reindex(columns=cols)
    df_n = df_n.reindex(columns=cols)

    df = pd.concat([df_b, df_n], ignore_index=True)
    if "mean_test_score" not in df.columns:
        raise ValueError("mean_test_score not found in RF cv_results, cannot rank.")
    df["rank_test_score"] = df["mean_test_score"].rank(ascending=False, method="min").astype(int)

    save_cv_results(df, out_merged)
    return df

# =========================
# Coarse optimization + validation evaluation
# =========================
summary_rows, val_rows = [], []
best_params_all = {}

# ---- DecisionTree ----
print_section("Optimizing DecisionTree (CV on TRAIN only)")
dt_search = run_random_search(models["DecisionTree"], param_spaces["DecisionTree"], N_ITER, SEED, OUT_CV_DT)
dt_best = dt_search.best_estimator_
dt_val_pred = dt_best.predict(X_val)
dt_val_mae, dt_val_rmse, dt_val_r2 = evaluate_regression(y_val, dt_val_pred)
joblib.dump(dt_best, MODEL_PATH_DT)

best_params_all["DecisionTree"] = dt_search.best_params_
summary_rows.append({"Model": "DecisionTree", "BestParams_CVTrain": dt_search.best_params_,
                     "BestModelPath": str(MODEL_PATH_DT), "Val_MAE": dt_val_mae, "Val_RMSE": dt_val_rmse, "Val_R2": dt_val_r2})
val_rows.append({"Model": "DecisionTree", "Val_MAE": dt_val_mae, "Val_RMSE": dt_val_rmse, "Val_R2": dt_val_r2})

print(f"Best CV params: {dt_search.best_params_}")
print(f"Validation MAE:  {dt_val_mae:.3f} | RMSE: {dt_val_rmse:.3f} | R2: {dt_val_r2:.3f}")
print(f"Saved model to:   {MODEL_PATH_DT.resolve()}")

# ---- ExtraTrees ----
print_section("Optimizing ExtraTrees (CV on TRAIN only)")
et_search = run_random_search(models["ExtraTrees"], param_spaces["ExtraTrees"], N_ITER, SEED, OUT_CV_ET)
et_best = et_search.best_estimator_
et_val_pred = et_best.predict(X_val)
et_val_mae, et_val_rmse, et_val_r2 = evaluate_regression(y_val, et_val_pred)
joblib.dump(et_best, MODEL_PATH_ET)

best_params_all["ExtraTrees"] = et_search.best_params_
summary_rows.append({"Model": "ExtraTrees", "BestParams_CVTrain": et_search.best_params_,
                     "BestModelPath": str(MODEL_PATH_ET), "Val_MAE": et_val_mae, "Val_RMSE": et_val_rmse, "Val_R2": et_val_r2})
val_rows.append({"Model": "ExtraTrees", "Val_MAE": et_val_mae, "Val_RMSE": et_val_rmse, "Val_R2": et_val_r2})

print(f"Best CV params: {et_search.best_params_}")
print(f"Validation MAE:  {et_val_mae:.3f} | RMSE: {et_val_rmse:.3f} | R2: {et_val_r2:.3f}")
print(f"Saved model to:   {MODEL_PATH_ET.resolve()}")

# ---- RandomForest (two-branch search) ----
print_section("Optimizing RandomForest (bootstrap vs no-bootstrap)")
n_iter_half = max(10, N_ITER // 2)

print("\n--- RF Search 1/2: bootstrap=True (max_samples enabled) ---")
rf_search_boot = run_random_search(models["RandomForest"], rf_space_bootstrap, n_iter_half, SEED, OUT_CV_RF_BOOT)

print("\n--- RF Search 2/2: bootstrap=False (max_samples disabled) ---")
rf_search_no = run_random_search(models["RandomForest"], rf_space_no_bootstrap, n_iter_half, SEED + 1, OUT_CV_RF_NOBOOT)

# Select best by CV score (neg-MAE; higher is better)
best_rf_search = rf_search_boot if rf_search_boot.best_score_ >= rf_search_no.best_score_ else rf_search_no
rf_best = best_rf_search.best_estimator_

# Auto-merge two RF CV files into one stable file for downstream refine
_ = merge_rf_cv(OUT_CV_RF_BOOT, OUT_CV_RF_NOBOOT, OUT_CV_RF_MERGED)

rf_val_pred = rf_best.predict(X_val)
rf_val_mae, rf_val_rmse, rf_val_r2 = evaluate_regression(y_val, rf_val_pred)
joblib.dump(rf_best, MODEL_PATH_RF)

best_params_all["RandomForest"] = best_rf_search.best_params_
summary_rows.append({"Model": "RandomForest", "BestParams_CVTrain": best_rf_search.best_params_,
                     "BestModelPath": str(MODEL_PATH_RF), "Val_MAE": rf_val_mae, "Val_RMSE": rf_val_rmse, "Val_R2": rf_val_r2})
val_rows.append({"Model": "RandomForest", "Val_MAE": rf_val_mae, "Val_RMSE": rf_val_rmse, "Val_R2": rf_val_r2})

print(f"\nSelected RF branch: {'bootstrap=True' if best_rf_search is rf_search_boot else 'bootstrap=False'}")
print(f"Best CV params: {best_rf_search.best_params_}")
print(f"Validation MAE:  {rf_val_mae:.3f} | RMSE: {rf_val_rmse:.3f} | R2: {rf_val_r2:.3f}")
print(f"Saved model to:   {MODEL_PATH_RF.resolve()}")

# ---------- Save summary tables ----------
df_summary = pd.DataFrame(summary_rows).sort_values("Val_MAE")
df_val_rank = pd.DataFrame(val_rows).sort_values("Val_MAE")
df_summary.to_csv(OUT_SUMMARY, index=False)
df_val_rank.to_csv(OUT_VAL_RANK, index=False)

# Save best params (for paper appendix / future reuse)
with open(BEST_PARAMS_JSON, "w", encoding="utf-8") as f:
    json.dump(best_params_all, f, indent=2)

print_section("Validation Ranking (model selection; WITH shielding)")
print(f"Saved summary table: {OUT_SUMMARY.resolve()}")
print(f"Saved val ranking : {OUT_VAL_RANK.resolve()}")
print(f"Saved best params : {BEST_PARAMS_JSON.resolve()}")
display(prepare_display_table(df_val_rank))


### 6. Refined hyperparameter search without shielding

Purpose: Refine the top-performing structure-only models and record their validation ranking for downstream selection.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 6. Refined hyperparameter search without shielding
# Purpose: Refine the top-performing structure-only models and record their validation ranking for downstream selection.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import ast

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

# ---------- Reproducibility ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
CV_FOLDS = 5

# Refinement controls
TOPK_SEEDS = 5                 # how many coarse "best" configs to seed refinement
N_ITER_PER_SEED = 40           # randomized iterations per seed (total per model = TOPK_SEEDS*N_ITER_PER_SEED)
CAT_TOPN = 2                   # keep top-N categorical values (based on coarse top rows) + best
K_NEIGHBORS = 2                # for ordered list params
INT_WINDOW = 3                 # refine int params as best±window
FLOAT_RATIO = 0.5              # refine float params as best*(1±ratio)
RANDOM_STATE_BASE = 1000       # to vary random_state across seeds while reproducible

# ---------- Paths ----------
PROJECT_ROOT = Path(".").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
TABLE_DIR = RESULTS_DIR / "tables"
MODEL_DIR = RESULTS_DIR / "models"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "features_dataset.parquet"

# IMPORTANT: coarse CV outputs from Cell 12 (use the merged RF file you created)
CV_PATHS_COARSE = {
    "DecisionTree": TABLE_DIR / "cv_results_decisiontree.csv",
    "RandomForest": TABLE_DIR / "cv_results_randomforest.csv",
    "ExtraTrees": TABLE_DIR / "cv_results_extratrees.csv",
}

# Outputs
OUT_REFINED_RANK = TABLE_DIR / "refined_validation_ranking.csv"
OUT_REFINED_CV = {
    "DecisionTree": TABLE_DIR / "cv_results_refined_decisiontree.csv",
    "RandomForest": TABLE_DIR / "cv_results_refined_randomforest.csv",
    "ExtraTrees": TABLE_DIR / "cv_results_refined_extratrees.csv",
}
OUT_REFINED_MODEL = {
    "DecisionTree": MODEL_DIR / "best_on_val_refined_DecisionTree.pkl",
    "RandomForest": MODEL_DIR / "best_on_val_refined_RandomForest.pkl",
    "ExtraTrees": MODEL_DIR / "best_on_val_refined_ExtraTrees.pkl",
}

# ---------- Checks ----------
if not FEATURE_PATH.exists():
    raise FileNotFoundError(f"Feature dataset not found: {FEATURE_PATH}")
for m, p in CV_PATHS_COARSE.items():
    if not p.exists():
        raise FileNotFoundError(f"Coarse CV results for {m} not found: {p} (run Cell 12 + merge RF first)")

# ---------- Load dataset ----------
df_features = pd.read_parquet(FEATURE_PATH)
required_cols = {"SMILES", "shift_value"}
missing_cols = required_cols - set(df_features.columns)
if missing_cols:
    raise ValueError(f"Feature table missing required columns: {missing_cols}")

X = df_features.drop(columns=["SMILES", "shift_value"])
y = df_features["shift_value"].astype(float)

# ---------- Reproduce Train/Val/Test split (must match Cell 12) ----------
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(1/9), random_state=SEED)

print_section("Split Summary (reproducibility; TEST not used in refinement selection)")
print(f"Total samples : {len(X)}")
print(f"Train samples : {len(X_train)}")
print(f"Val samples   : {len(X_val)}")
print(f"Test samples  : {len(X_test)}")
print(f"Feature dim   : {X.shape[1]}")

# ---------- Metrics ----------
def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

# =========================
# Coarse spaces (for local narrowing)
# Must match Cell 12 ranges
# =========================
param_spaces_coarse = {
    "DecisionTree": {
        "criterion": ["squared_error", "friedman_mse", "absolute_error"],
        "splitter": ["best", "random"],
        "max_depth": [None, 5, 10, 20, 30, 50, 80, 120],
        "min_samples_split": [2, 5, 10, 20, 50, 100],
        "min_samples_leaf": [1, 2, 4, 8, 16, 32],
        "max_features": [None, "sqrt", "log2", 0.2, 0.4, 0.6, 0.8],
        "min_impurity_decrease": [0.0, 1e-7, 1e-6, 1e-5, 1e-4],
        "ccp_alpha": [0.0, 1e-7, 1e-6, 1e-5, 1e-4],
    },
    "RandomForest": {
        "n_estimators": [200, 400, 800, 1200, 1600],
        "criterion": ["squared_error", "absolute_error", "friedman_mse"],
        "max_depth": [None, 10, 20, 40, 60, 80, 120],
        "min_samples_split": [2, 5, 10, 20, 50],
        "min_samples_leaf": [1, 2, 4, 8, 16],
        "max_features": ["sqrt", "log2", None, 0.2, 0.4, 0.6, 0.8],
        "bootstrap": [True, False],
        "max_samples": [None, 0.5, 0.7, 0.9],
        "min_impurity_decrease": [0.0, 1e-7, 1e-6, 1e-5],
        "ccp_alpha": [0.0, 1e-7, 1e-6, 1e-5],
    },
    "ExtraTrees": {
        "n_estimators": [200, 400, 800, 1200, 1600],
        "criterion": ["squared_error", "absolute_error", "friedman_mse"],
        "max_depth": [None, 10, 20, 40, 60, 80, 120],
        "min_samples_split": [2, 5, 10, 20, 50],
        "min_samples_leaf": [1, 2, 4, 8, 16],
        "max_features": ["sqrt", "log2", None, 0.2, 0.4, 0.6, 0.8],
        "min_impurity_decrease": [0.0, 1e-7, 1e-6, 1e-5],
        "ccp_alpha": [0.0, 1e-7, 1e-6, 1e-5],
    },
}

# =========================
# Helpers for refined space
# =========================
def _is_categorical(v):
    return isinstance(v, (str, bool)) or v is None

def _dedup_keep_order(seq):
    out = []
    for x in seq:
        if x not in out:
            out.append(x)
    return out

def _refine_from_ordered_list(values, best_value, k=2):
    if best_value not in values:
        return [best_value]
    idx = values.index(best_value)
    lo = max(0, idx - k)
    hi = min(len(values), idx + k + 1)
    return _dedup_keep_order(values[lo:hi])

def _refine_int(best_value, window=3, min_value=1):
    lo = max(min_value, int(best_value) - window)
    hi = max(lo + 1, int(best_value) + window)
    return list(range(lo, hi + 1))

def _refine_float(best_value, ratio=0.5, n_points=7):
    best_value = float(best_value)
    if best_value == 0.0:
        return [0.0]
    lo = best_value * (1.0 - ratio)
    hi = best_value * (1.0 + ratio)
    if lo > 0 and hi > 0:
        return list(np.exp(np.linspace(np.log(lo), np.log(hi), n_points)))
    return list(np.linspace(lo, hi, n_points))

def _smart_refine_max_features(coarse_values, best_value, k=2):
    """
    max_features often mixes {None, 'sqrt', 'log2', floats}.
    Strategy:
    - Always keep best_value
    - Keep all non-numeric categorical options (None/'sqrt'/'log2') seen in coarse_values (small set)
    - For numeric best_value -> keep numeric neighbors
    """
    cats = [v for v in coarse_values if isinstance(v, (str, type(None)))]
    nums = [v for v in coarse_values if isinstance(v, (int, float)) and not isinstance(v, bool)]

    out = []
    # keep best
    out.append(best_value)

    # keep categorical set (small)
    for v in cats:
        if v not in out:
            out.append(v)

    # if best is numeric, keep numeric neighbors; if best is categorical, keep a few numeric midpoints
    if isinstance(best_value, (int, float)) and not isinstance(best_value, bool):
        nums_sorted = sorted(nums)
        out += _refine_from_ordered_list(nums_sorted, float(best_value), k=k)
    else:
        # include a small subset of numeric defaults
        for v in [0.2, 0.4, 0.6]:
            if v in nums and v not in out:
                out.append(v)

    return _dedup_keep_order(out)

def top_categorical_candidates_from_coarse(cv_df, param_name, topk=20, keep_n=2):
    """
    Look at top-k rows (by rank_test_score) and collect most frequent values for a categorical param.
    Return up to keep_n values (ordered by frequency then first occurrence).
    """
    if "rank_test_score" not in cv_df.columns or "params" not in cv_df.columns:
        return None
    top = cv_df.sort_values("rank_test_score").head(topk)
    vals = []
    for s in top["params"].tolist():
        d = ast.literal_eval(s)
        if param_name in d:
            vals.append(d[param_name])
    if not vals:
        return None
    # frequency
    counts = {}
    first_idx = {}
    for i, v in enumerate(vals):
        counts[v] = counts.get(v, 0) + 1
        if v not in first_idx:
            first_idx[v] = i
    ranked = sorted(counts.keys(), key=lambda v: (-counts[v], first_idx[v]))
    return ranked[:keep_n]

def build_refined_space(model_name, best_params, coarse_space, cv_df_for_cats=None,
                        k_neighbors=2, int_window=3, float_ratio=0.5, cat_topn=2):
    """
    Improvements vs original:
    - Use coarse top rows to allow a SMALL set of categorical options (not fixed to single value).
    - Special handling for max_features mixed types.
    - RF bootstrap branch consistent: if best bootstrap is False, we drop max_samples.
    """
    refined = {}

    for p, best_v in best_params.items():
        # RF constraint: max_samples only when bootstrap=True
        if model_name == "RandomForest" and p == "max_samples":
            if best_params.get("bootstrap", False) is False:
                continue

        if p in coarse_space and isinstance(coarse_space[p], list):
            # max_features special
            if p == "max_features":
                refined[p] = _smart_refine_max_features(coarse_space[p], best_v, k=k_neighbors)
                continue

            # categorical: keep best + a couple of popular alternatives from coarse top rows
            if _is_categorical(best_v):
                cand = [best_v]
                if cv_df_for_cats is not None:
                    extra = top_categorical_candidates_from_coarse(cv_df_for_cats, p, topk=30, keep_n=cat_topn)
                    if extra:
                        for v in extra:
                            if v not in cand:
                                cand.append(v)
                refined[p] = cand
            else:
                refined[p] = _refine_from_ordered_list(coarse_space[p], best_v, k=k_neighbors)
            continue

        # fallback by type
        if _is_categorical(best_v):
            refined[p] = [best_v]
        elif isinstance(best_v, int):
            refined[p] = _refine_int(best_v, window=int_window, min_value=1)
        elif isinstance(best_v, float):
            refined[p] = _refine_float(best_v, ratio=float_ratio, n_points=7)
        else:
            refined[p] = [best_v]

    # Final RF safety: enforce bootstrap branch consistency
    if model_name == "RandomForest":
        boot = best_params.get("bootstrap", None)
        if boot is not None:
            refined["bootstrap"] = [boot]  # FIX branch
            if boot is False:
                refined.pop("max_samples", None)

    return refined

# =========================
# Model factories
# =========================
def make_estimator(model_name):
    if model_name == "DecisionTree":
        return DecisionTreeRegressor(random_state=SEED)
    if model_name == "RandomForest":
        return RandomForestRegressor(random_state=SEED, n_jobs=20)
    if model_name == "ExtraTrees":
        return ExtraTreesRegressor(random_state=SEED, n_jobs=20)
    raise ValueError(f"Unknown model: {model_name}")

# =========================
# Main refinement (Top-K seeds)
# =========================
refined_rank_rows = []

for model_name in ["DecisionTree", "RandomForest", "ExtraTrees"]:
    print(f"\n==============================")
    print(f"Refinement: {model_name} (Top-{TOPK_SEEDS} seeds)")
    print(f"==============================")

    cv_df = pd.read_csv(CV_PATHS_COARSE[model_name])

    if ("rank_test_score" not in cv_df.columns) or ("params" not in cv_df.columns):
        raise ValueError(f"{CV_PATHS_COARSE[model_name]} missing required columns: rank_test_score / params")

    # ---- Take Top-K coarse seeds (by rank_test_score) ----
    cv_top = cv_df.sort_values("rank_test_score").head(TOPK_SEEDS).copy()
    seed_params_list = [ast.literal_eval(s) for s in cv_top["params"].tolist()]

    all_refined_results = []
    best_refined_estimator = None
    best_refined_cv_score = -np.inf  # higher neg-MAE is better

    # We'll collect all cv_results_ across seeds (tagged) for auditing/paper
    cv_results_all_seeds = []

    for seed_idx, best_params in enumerate(seed_params_list, start=1):
        print(f"\n--- Seed {seed_idx}/{TOPK_SEEDS}: coarse-best params ---")
        print(prepare_display_table(pd.DataFrame([best_params])))

        # Build refined space around this seed
        refined_space = build_refined_space(
            model_name=model_name,
            best_params=best_params,
            coarse_space=param_spaces_coarse[model_name],
            cv_df_for_cats=cv_df,
            k_neighbors=K_NEIGHBORS,
            int_window=INT_WINDOW,
            float_ratio=FLOAT_RATIO,
            cat_topn=CAT_TOPN,
        )

        # Quick summary
        print("Refined space keys:", list(refined_space.keys()))
        print("Refined space sizes:", {k: len(v) for k, v in refined_space.items()})

        estimator = make_estimator(model_name)

        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=refined_space,
            n_iter=N_ITER_PER_SEED,
            cv=CV_FOLDS,
            scoring="neg_mean_absolute_error",
            n_jobs=-1,
            random_state=RANDOM_STATE_BASE + seed_idx,  # vary per seed but reproducible
            verbose=1,
            return_train_score=True,
        )
        search.fit(X_train, y_train)

        # Track best across seeds by CV score (TRAIN CV)
        if search.best_score_ > best_refined_cv_score:
            best_refined_cv_score = search.best_score_
            best_refined_estimator = search.best_estimator_

        # Store detailed results
        df_seed = pd.DataFrame(search.cv_results_)
        df_seed["SeedIndex"] = seed_idx
        df_seed["CoarseSeedParams"] = str(best_params)
        cv_results_all_seeds.append(df_seed)

        all_refined_results.append({
            "SeedIndex": seed_idx,
            "BestScore_CVTrain_negMAE": search.best_score_,
            "BestParams_CVTrain": search.best_params_,
        })

        print(f"Seed {seed_idx} best CV neg-MAE: {search.best_score_:.6f}")
        print(f"Seed {seed_idx} best params: {search.best_params_}")

    # ---- Save merged refined CV results (all seeds) ----
    df_cv_refined_all = pd.concat(cv_results_all_seeds, ignore_index=True)
    df_cv_refined_all.to_csv(OUT_REFINED_CV[model_name], index=False)
    print(f"\nSaved refined CV results (ALL seeds) to:\n  {OUT_REFINED_CV[model_name].resolve()}")

    # ---- Evaluate the globally best refined estimator on VAL ----
    y_val_pred = best_refined_estimator.predict(X_val)
    val_mae, val_rmse, val_r2 = evaluate_regression(y_val, y_val_pred)

    # Save refined model
    joblib.dump(best_refined_estimator, OUT_REFINED_MODEL[model_name])
    print(f"Saved refined best model to:\n  {OUT_REFINED_MODEL[model_name].resolve()}")

    # Record for ranking
    refined_rank_rows.append({
        "Model": model_name,
        "Val_MAE": val_mae,
        "Val_RMSE": val_rmse,
        "Val_R2": val_r2,
        "BestRefined_CVTrain_negMAE": float(best_refined_cv_score),
        "RefinedModelPath": str(OUT_REFINED_MODEL[model_name]),
        "RefinedCVPath": str(OUT_REFINED_CV[model_name]),
        "TopKSeedsUsed": TOPK_SEEDS,
        "NIterPerSeed": N_ITER_PER_SEED,
        "TotalItersApprox": TOPK_SEEDS * N_ITER_PER_SEED,
    })

    print("\nValidation performance (Best Refined across seeds):")
    print(f"Val MAE : {val_mae:.3f}")
    print(f"Val RMSE: {val_rmse:.3f}")
    print(f"Val R2  : {val_r2:.3f}")

# ---- Save refined ranking ----
df_refined_rank = pd.DataFrame(refined_rank_rows).sort_values("Val_MAE")
df_refined_rank.to_csv(OUT_REFINED_RANK, index=False)

print_section("Refined Validation Ranking (ALL models; Top-K seeds refinement)")
print(f"Saved to: {OUT_REFINED_RANK.resolve()}")
display(prepare_display_table(df_refined_rank))


### 7. Exploratory analysis of shielding values

Purpose: Visualize the shielding distribution and its relationship to the 19F chemical shift.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 7. Exploratory analysis of shielding values
# Purpose: Visualize the shielding distribution and its relationship to the 19F chemical shift.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Input check ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('.').resolve()

if 'df_cleaned' not in globals():
    CLEANED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_shift_data_with_shield.csv'
    if not CLEANED_PATH.exists():
        raise FileNotFoundError(
            'df_cleaned not found and cleaned file not found. '
            'Please run Cell 8 first or ensure cleaned_shift_data_with_shield.csv exists.'
        )
    df_cleaned = pd.read_csv(CLEANED_PATH)

required_cols = {'SMILES', 'shift_value', 'shielding_value'}
missing = required_cols - set(df_cleaned.columns)
if missing:
    raise ValueError(f'df_cleaned missing required columns: {missing}')

df_cleaned['shift_value'] = pd.to_numeric(df_cleaned['shift_value'], errors='coerce')
df_cleaned['shielding_value'] = pd.to_numeric(df_cleaned['shielding_value'], errors='coerce')
df_plot = df_cleaned.dropna(subset=['shift_value', 'shielding_value']).copy()

RESULTS_DIR = PROJECT_ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

FIG_SHIFT_HIST = FIG_DIR / 'Fig_shift_distribution.png'
FIG_SHIELD_SCATTER = FIG_DIR / 'Fig_shielding_vs_shift.png'

apply_plot_style()

print_section('Shift-distribution figure')
fig, ax = create_single_axis_figure('single_wide')

x = df_plot['shift_value'].to_numpy()
q25, q75 = np.percentile(x, [25, 75])
iqr = q75 - q25
bin_width = 2 * iqr * (len(x) ** (-1 / 3)) if (iqr > 0 and len(x) > 1) else None
if bin_width and bin_width > 0:
    n_bins = int(np.ceil((x.max() - x.min()) / bin_width)) if x.max() > x.min() else 10
    n_bins = max(10, min(n_bins, 80))
else:
    n_bins = 30

ax.hist(df_plot['shift_value'], bins=n_bins, edgecolor='black', linewidth=0.6)
ax.set_xlabel(r'$^{19}$F NMR chemical shift (ppm)')
ax.set_ylabel('Count')
ax.set_title(r'Distribution of $^{19}$F NMR chemical shifts')
style_axes(ax)

fig.tight_layout()
saved_paths = save_figure(fig, FIG_SHIFT_HIST)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')

print_section('Shielding-versus-shift figure')
fig, ax = create_single_axis_figure('single_wide')

ax.scatter(
    df_plot['shielding_value'],
    df_plot['shift_value'],
    s=18,
    alpha=0.75,
    edgecolors='none',
)

ax.set_xlabel('Magnetic shielding value (a.u.)')
ax.set_ylabel(r'$^{19}$F NMR chemical shift (ppm)')
ax.set_title('Shielding value versus chemical shift')

xv = df_plot['shielding_value'].to_numpy()
yv = df_plot['shift_value'].to_numpy()
if len(xv) >= 2 and np.std(xv) > 0:
    coef = np.polyfit(xv, yv, 1)
    xx = np.linspace(xv.min(), xv.max(), 200)
    yy = coef[0] * xx + coef[1]
    ax.plot(xx, yy, linewidth=1.4, color='#D62728')

style_axes(ax)

fig.tight_layout()
saved_paths = save_figure(fig, FIG_SHIELD_SCATTER)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')

print_section('Plotting Summary')
print(f'Samples used for plotting: {len(df_plot)}')
print(f"shift_value range (ppm)   : {df_plot['shift_value'].min():.3f} to {df_plot['shift_value'].max():.3f}")
print(f"shielding_value range     : {df_plot['shielding_value'].min():.6f} to {df_plot['shielding_value'].max():.6f}")


### 8. Molecular feature construction with shielding

Purpose: Augment the descriptor matrix with shielding values while preserving the established feature-engineering pipeline.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 8. Molecular feature construction with shielding
# Purpose: Augment the descriptor matrix with shielding values while preserving the established feature-engineering pipeline.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import MACCSkeys, Descriptors
from rdkit import DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

# ---------- Input check ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(".").resolve()

if "df_cleaned" not in globals():
    raise RuntimeError("df_cleaned not found. Please run Cell 8 (data cleaning) first.")

required_cols = {"SMILES", "shift_value", "shielding_value"}
missing_cols = required_cols - set(df_cleaned.columns)
if missing_cols:
    raise ValueError(f"df_cleaned missing columns: {missing_cols}")

# Ensure numeric (drop rows with missing values)
df_cleaned = df_cleaned.copy()
df_cleaned["shift_value"] = pd.to_numeric(df_cleaned["shift_value"], errors="coerce")
df_cleaned["shielding_value"] = pd.to_numeric(df_cleaned["shielding_value"], errors="coerce")
df_cleaned = df_cleaned.dropna(subset=["shift_value", "shielding_value"]).reset_index(drop=True)

# ---------- Paths ----------
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = PROCESSED_DIR / "features_dataset_with_shield.csv"
OUT_PARQUET = PROCESSED_DIR / "features_dataset_with_shield.parquet"

# ---------- Parameters ----------
N_BITS = 1024
MORGAN_RADIUS = 2
ECFP_RADIUS = 4

# ---------- Fingerprint generators (RDKit-compatible signature) ----------
# IMPORTANT:
# Your RDKit build supports includeChirality, but does NOT accept includeFeatures.
morgan_gen = GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=N_BITS, includeChirality=False)
ecfp_gen = GetMorganGenerator(radius=ECFP_RADIUS, fpSize=N_BITS, includeChirality=True)

# =========================
# Helper functions
# =========================

def mol_from_smiles(smiles: str):
    """Convert SMILES to RDKit Mol object. Returns None if parsing fails."""
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

def bitvect_to_numpy(fp) -> np.ndarray:
    """Convert RDKit ExplicitBitVect to a numpy array of 0/1."""
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def compute_basic_descriptors(mol: Chem.Mol) -> dict:
    """Compute basic physicochemical descriptors (numeric)."""
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
        "RingCount": Descriptors.RingCount(mol),
    }

def compute_morgan_fp(mol: Chem.Mol) -> np.ndarray:
    """Compute Morgan fingerprint (1024 bits) using MorganGenerator."""
    fp = morgan_gen.GetFingerprint(mol)
    return bitvect_to_numpy(fp)

def compute_ecfp_per_fluorine_or(mol: Chem.Mol) -> np.ndarray:
    """
    Compute fluorine-centered ECFP-like fingerprints (1024 bits).
    For multiple F atoms, combine fingerprints using bitwise OR (max).

    NOTE:
    This uses MorganGenerator with fromAtoms selection.
    It does NOT enable 'feature Morgan' to keep compatibility with your RDKit build.
    """
    fluorines = [a for a in mol.GetAtoms() if a.GetSymbol() == "F"]
    if not fluorines:
        return np.zeros((N_BITS,), dtype=np.int8)

    combined = np.zeros((N_BITS,), dtype=np.int8)
    for atom in fluorines:
        fp = ecfp_gen.GetFingerprint(mol, fromAtoms=[atom.GetIdx()])
        combined = np.maximum(combined, bitvect_to_numpy(fp))
    return combined

def compute_maccs_fp(mol: Chem.Mol) -> np.ndarray:
    """
    Compute MACCS keys.
    RDKit typically returns 167 bits (including bit 0). Keep native length.
    """
    fp = MACCSkeys.GenMACCSKeys(mol)
    return bitvect_to_numpy(fp)

# =========================
# Feature construction
# =========================

feature_rows = []
invalid_smiles_count = 0

for _, row in df_cleaned.iterrows():
    smiles = str(row["SMILES"]).strip()
    shift = float(row["shift_value"])
    shield = float(row["shielding_value"])

    mol = mol_from_smiles(smiles)
    if mol is None:
        invalid_smiles_count += 1
        continue

    basic_desc = compute_basic_descriptors(mol)
    morgan_fp = compute_morgan_fp(mol)
    ecfp_f_fp = compute_ecfp_per_fluorine_or(mol)
    maccs_fp = compute_maccs_fp(mol)

    # Core columns
    feature_dict = {
        "SMILES": smiles,
        "shift_value": shift,
        "shielding_value": shield,
        **basic_desc,
    }

    # Morgan bits
    for i, bit in enumerate(morgan_fp):
        feature_dict[f"Morgan_{i}"] = int(bit)

    # F-centered ECFP-like bits
    for i, bit in enumerate(ecfp_f_fp):
        feature_dict[f"ECFP_F_{i}"] = int(bit)

    # MACCS bits (native length)
    for i, bit in enumerate(maccs_fp):
        feature_dict[f"MACCS_{i}"] = int(bit)

    feature_rows.append(feature_dict)

df_features = pd.DataFrame(feature_rows)

# =========================
# Save dataset (CSV required; Parquet optional)
# =========================

df_features.to_csv(OUT_CSV, index=False)

try:
    df_features.to_parquet(OUT_PARQUET, index=False)
    parquet_msg = f"Saved Parquet to: {OUT_PARQUET}"
except Exception as e:
    parquet_msg = f"Parquet not saved (optional). Reason: {type(e).__name__}: {e}"

# =========================
# Summary
# =========================

n_desc = 7
n_morgan = N_BITS
n_ecfp = N_BITS
n_maccs = len([c for c in df_features.columns if c.startswith("MACCS_")])

print_section("Feature Engineering Summary (with shielding_value)")
print(f"Total input samples (df_cleaned) : {len(df_cleaned)}")
print(f"Invalid SMILES removed           : {invalid_smiles_count}")
print(f"Final dataset size               : {len(df_features)}")

print("\n----- Feature dimensions -----")
print(f"shielding_value                  : 1")
print(f"PhysChem descriptors             : {n_desc}")
print(f"Morgan fingerprint               : {n_morgan}")
print(f"Fluorine-centered ECFP-like (OR) : {n_ecfp}")
print(f"MACCS keys (native length)       : {n_maccs}")
print(f"Total feature dimension          : {df_features.shape[1] - 2}")  # excluding SMILES & shift_value

print("\n----- Output files -----")
print(f"Saved CSV to                     : {OUT_CSV}")
print_section('Saved feature files')
print(parquet_msg)

display(prepare_display_table(df_features.head()))


### 9. Baseline model benchmark with shielding

Purpose: Compare standard regression models after including shielding information in the feature set.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 9. Baseline model benchmark with shielding
# Purpose: Compare standard regression models after including shielding information in the feature set.
# -----------------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42

PROJECT_ROOT = Path('.').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
TABLE_DIR = RESULTS_DIR / 'tables'
FIG_DIR = RESULTS_DIR / 'figures'

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

OUT_TABLE = TABLE_DIR / 'model_benchmark_results_with_shield.csv'
OUT_FIG = FIG_DIR / 'model_benchmark_mae.png'

apply_plot_style()

FEATURE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'features_dataset_with_shield.parquet'
if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        f'Feature dataset not found: {FEATURE_PATH}\n'
        'Please run Cell 10 first to generate features with shielding_value.'
    )

df_features = pd.read_parquet(FEATURE_PATH)

X = df_features.drop(columns=['SMILES', 'shift_value'])
y = df_features['shift_value'].astype(float)

print_section('Dataset Summary')
print(f'Total samples : {len(X)}')
print(f'Feature dim   : {X.shape[1]}')
print(f"shielding_value included in X? : {'shielding_value' in X.columns}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=SEED
)

print(f'Train samples : {len(X_train)}')
print(f'Test samples  : {len(X_test)}')

models = {
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'DecisionTree': DecisionTreeRegressor(random_state=SEED),
    'RandomForest': RandomForestRegressor(random_state=SEED),
    'ExtraTrees': ExtraTreesRegressor(random_state=SEED),
    'GradientBoosting': GradientBoostingRegressor(random_state=SEED),
    'AdaBoost': AdaBoostRegressor(random_state=SEED),
    'SVR': SVR(),
    'KNN': KNeighborsRegressor(),
}

results = []

for name, model in models.items():
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
        r2 = r2_score(y_test, y_pred)

        results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})

        print(metric_row(name, mae, rmse, r2))

    except Exception as e:
        print(f'{name} failed: {type(e).__name__}: {e}')

df_results = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)

df_results.to_csv(OUT_TABLE, index=False)

report_saved_paths([OUT_TABLE], 'Saved result tables')

display(prepare_display_table(df_results))

fig, ax = create_single_axis_figure('single_wide')
ax.bar(df_results['Model'], df_results['MAE'], color='#4C78A8')
ax.tick_params(axis='x', rotation=90)
ax.set_ylabel('MAE (ppm)')
ax.set_title('Benchmark of regression models with shielding')
style_axes(ax, grid_axis='y')
fig.tight_layout()

saved_paths = save_figure(fig, OUT_FIG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### 10. Coarse hyperparameter search with shielding

Purpose: Run the initial train/validation model-selection step for the shielding-augmented workflow.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 10. Coarse hyperparameter search with shielding
# Purpose: Run the initial train/validation model-selection step for the shielding-augmented workflow.
# -----------------------------------------------------------------------------

from pathlib import Path
import os, json, random, csv
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

# ---------- Reproducibility ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
N_ITER = 100
CV_FOLDS = 5
SEARCH_NJOBS = 20

def set_global_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_global_seed(SEED)

# ---------- Paths ----------
PROJECT_ROOT = Path(".").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
TABLE_DIR = RESULTS_DIR / "tables"
MODEL_DIR = RESULTS_DIR / "models"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Distinct filenames for WITH-shield pipeline
OUT_SUMMARY = TABLE_DIR / "optimized_model_results_train_val_test_with_shield.csv"
OUT_VAL_RANK = TABLE_DIR / "validation_ranking_with_shield.csv"
OUT_TEST_FINAL = TABLE_DIR / "final_test_performance_with_shield.csv"

OUT_CV_DT = TABLE_DIR / "cv_results_decisiontree_with_shield.csv"
OUT_CV_ET = TABLE_DIR / "cv_results_extratrees_with_shield.csv"

# RF will be generated as two files then auto-merged into OUT_CV_RF_MERGED
OUT_CV_RF_BOOT = TABLE_DIR / "cv_results_randomforest_with_shield_bootstrap.csv"
OUT_CV_RF_NOBOOT = TABLE_DIR / "cv_results_randomforest_with_shield_no_bootstrap.csv"
OUT_CV_RF_MERGED = TABLE_DIR / "cv_results_randomforest_with_shield.csv"  # <- used by later Cell 13

# Save models
MODEL_PATH_DT = MODEL_DIR / "best_on_val_DecisionTree_with_shield.pkl"
MODEL_PATH_RF = MODEL_DIR / "best_on_val_RandomForest_with_shield.pkl"
MODEL_PATH_ET = MODEL_DIR / "best_on_val_ExtraTrees_with_shield.pkl"
FINAL_MODEL_OUT = MODEL_DIR / "final_best_model_with_shield.pkl"

# Also save best params (paper appendix / reproducibility)
BEST_PARAMS_JSON = TABLE_DIR / "best_params_with_shield.json"

# ---------- Load dataset ----------
FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "features_dataset_with_shield.parquet"
if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Feature dataset not found: {FEATURE_PATH}\n"
        "Please run the feature-engineering cell for shift_data_with_shield first."
    )

df_features = pd.read_parquet(FEATURE_PATH)

required_cols = {"SMILES", "shift_value", "shielding_value"}
missing = required_cols - set(df_features.columns)
if missing:
    raise ValueError(
        f"Missing required columns in features file: {missing}\n"
        "Make sure shielding_value is kept as a numeric feature."
    )

X = df_features.drop(columns=["SMILES", "shift_value"])
y = df_features["shift_value"].astype(float)

# ---------- Train / Validation / Test split ----------
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=(1 / 9), random_state=SEED
)

print_section("Dataset Summary (WITH shielding_value)")
print(f"Total samples : {len(X)}")
print(f"Train samples : {len(X_train)}")
print(f"Val samples   : {len(X_val)}")
print(f"Test samples  : {len(X_test)}")
print(f"Feature dim   : {X.shape[1]}")

# ---------- Parameter spaces (Coarse) ----------
param_spaces = {
    "DecisionTree": {
        "criterion": ["squared_error", "friedman_mse", "absolute_error"],
        "splitter": ["best", "random"],
        "max_depth": [None, 5, 10, 20, 30, 50, 80, 120],
        "min_samples_split": [2, 5, 10, 20, 50, 100],
        "min_samples_leaf": [1, 2, 4, 8, 16, 32],
        "max_features": [None, "sqrt", "log2", 0.2, 0.4, 0.6, 0.8],
        "min_impurity_decrease": [0.0, 1e-7, 1e-6, 1e-5, 1e-4],
        "ccp_alpha": [0.0, 1e-7, 1e-6, 1e-5, 1e-4],
    },
    "ExtraTrees": {
        "n_estimators": [200, 400, 800, 1200, 1600],
        "criterion": ["squared_error", "absolute_error", "friedman_mse"],
        "max_depth": [None, 10, 20, 40, 60, 80, 120],
        "min_samples_split": [2, 5, 10, 20, 50],
        "min_samples_leaf": [1, 2, 4, 8, 16],
        "max_features": ["sqrt", "log2", None, 0.2, 0.4, 0.6, 0.8],
        "min_impurity_decrease": [0.0, 1e-7, 1e-6, 1e-5],
        "ccp_alpha": [0.0, 1e-7, 1e-6, 1e-5],
    },
}

rf_common = {
    "n_estimators": [200, 400, 800, 1200, 1600],
    "criterion": ["squared_error", "absolute_error", "friedman_mse"],
    "max_depth": [None, 10, 20, 40, 60, 80, 120],
    "min_samples_split": [2, 5, 10, 20, 50],
    "min_samples_leaf": [1, 2, 4, 8, 16],
    "max_features": ["sqrt", "log2", None, 0.2, 0.4, 0.6, 0.8],
    "min_impurity_decrease": [0.0, 1e-7, 1e-6, 1e-5],
    "ccp_alpha": [0.0, 1e-7, 1e-6, 1e-5],
}

rf_space_bootstrap = {**rf_common, "bootstrap": [True], "max_samples": [0.5, 0.7, 0.9]}
rf_space_no_bootstrap = {**rf_common, "bootstrap": [False]}

# ---------- Models ----------
models = {
    "DecisionTree": DecisionTreeRegressor(random_state=SEED),
    "ExtraTrees": ExtraTreesRegressor(random_state=SEED, n_jobs=SEARCH_NJOBS),
    "RandomForest": RandomForestRegressor(random_state=SEED, n_jobs=SEARCH_NJOBS),
}

# ---------- Helpers ----------
def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

def save_cv_results(df_cv: pd.DataFrame, out_path: Path):
    """Write CV results robustly (params may contain commas)."""
    df_cv.to_csv(out_path, index=False, quoting=csv.QUOTE_MINIMAL, escapechar="\\")
    print(f"CV results saved to: {out_path.resolve()}")

def run_random_search(estimator, param_space, n_iter, seed, out_cv_path):
    """RandomizedSearchCV on TRAIN only (KFold CV), optimizing neg-MAE."""
    cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=seed)
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_space,
        n_iter=n_iter,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=SEARCH_NJOBS,
        random_state=seed,
        verbose=2,
        return_train_score=True
    )
    search.fit(X_train, y_train)
    save_cv_results(pd.DataFrame(search.cv_results_), out_cv_path)
    return search

def merge_rf_cv(boot_path: Path, noboot_path: Path, out_merged: Path):
    """Merge two RF CV results and recompute global ranking by mean_test_score (higher is better)."""
    df_b = pd.read_csv(boot_path, engine="python")
    df_n = pd.read_csv(noboot_path, engine="python")
    df_b["rf_branch"] = "bootstrap_true"
    df_n["rf_branch"] = "bootstrap_false"

    cols = sorted(set(df_b.columns).union(set(df_n.columns)))
    df_b = df_b.reindex(columns=cols)
    df_n = df_n.reindex(columns=cols)

    df = pd.concat([df_b, df_n], ignore_index=True)
    if "mean_test_score" not in df.columns:
        raise ValueError("mean_test_score not found in RF cv_results, cannot rank.")
    df["rank_test_score"] = df["mean_test_score"].rank(ascending=False, method="min").astype(int)

    save_cv_results(df, out_merged)
    return df

# =========================
# Coarse optimization + validation evaluation
# =========================
summary_rows, val_rows = [], []
best_params_all = {}

# ---- DecisionTree ----
print_section("Optimizing DecisionTree (CV on TRAIN only)")
dt_search = run_random_search(models["DecisionTree"], param_spaces["DecisionTree"], N_ITER, SEED, OUT_CV_DT)
dt_best = dt_search.best_estimator_
dt_val_pred = dt_best.predict(X_val)
dt_val_mae, dt_val_rmse, dt_val_r2 = evaluate_regression(y_val, dt_val_pred)
joblib.dump(dt_best, MODEL_PATH_DT)

best_params_all["DecisionTree"] = dt_search.best_params_
summary_rows.append({"Model": "DecisionTree", "BestParams_CVTrain": dt_search.best_params_,
                     "BestModelPath": str(MODEL_PATH_DT), "Val_MAE": dt_val_mae, "Val_RMSE": dt_val_rmse, "Val_R2": dt_val_r2})
val_rows.append({"Model": "DecisionTree", "Val_MAE": dt_val_mae, "Val_RMSE": dt_val_rmse, "Val_R2": dt_val_r2})

print(f"Best CV params: {dt_search.best_params_}")
print(f"Validation MAE:  {dt_val_mae:.3f} | RMSE: {dt_val_rmse:.3f} | R2: {dt_val_r2:.3f}")
print(f"Saved model to:   {MODEL_PATH_DT.resolve()}")

# ---- ExtraTrees ----
print_section("Optimizing ExtraTrees (CV on TRAIN only)")
et_search = run_random_search(models["ExtraTrees"], param_spaces["ExtraTrees"], N_ITER, SEED, OUT_CV_ET)
et_best = et_search.best_estimator_
et_val_pred = et_best.predict(X_val)
et_val_mae, et_val_rmse, et_val_r2 = evaluate_regression(y_val, et_val_pred)
joblib.dump(et_best, MODEL_PATH_ET)

best_params_all["ExtraTrees"] = et_search.best_params_
summary_rows.append({"Model": "ExtraTrees", "BestParams_CVTrain": et_search.best_params_,
                     "BestModelPath": str(MODEL_PATH_ET), "Val_MAE": et_val_mae, "Val_RMSE": et_val_rmse, "Val_R2": et_val_r2})
val_rows.append({"Model": "ExtraTrees", "Val_MAE": et_val_mae, "Val_RMSE": et_val_rmse, "Val_R2": et_val_r2})

print(f"Best CV params: {et_search.best_params_}")
print(f"Validation MAE:  {et_val_mae:.3f} | RMSE: {et_val_rmse:.3f} | R2: {et_val_r2:.3f}")
print(f"Saved model to:   {MODEL_PATH_ET.resolve()}")

# ---- RandomForest (two-branch search) ----
print_section("Optimizing RandomForest (bootstrap vs no-bootstrap)")
n_iter_half = max(10, N_ITER // 2)

print("\n--- RF Search 1/2: bootstrap=True (max_samples enabled) ---")
rf_search_boot = run_random_search(models["RandomForest"], rf_space_bootstrap, n_iter_half, SEED, OUT_CV_RF_BOOT)

print("\n--- RF Search 2/2: bootstrap=False (max_samples disabled) ---")
rf_search_no = run_random_search(models["RandomForest"], rf_space_no_bootstrap, n_iter_half, SEED + 1, OUT_CV_RF_NOBOOT)

# Select best by CV score (neg-MAE; higher is better)
best_rf_search = rf_search_boot if rf_search_boot.best_score_ >= rf_search_no.best_score_ else rf_search_no
rf_best = best_rf_search.best_estimator_

# Auto-merge two RF CV files into one stable file for downstream refine
_ = merge_rf_cv(OUT_CV_RF_BOOT, OUT_CV_RF_NOBOOT, OUT_CV_RF_MERGED)

rf_val_pred = rf_best.predict(X_val)
rf_val_mae, rf_val_rmse, rf_val_r2 = evaluate_regression(y_val, rf_val_pred)
joblib.dump(rf_best, MODEL_PATH_RF)

best_params_all["RandomForest"] = best_rf_search.best_params_
summary_rows.append({"Model": "RandomForest", "BestParams_CVTrain": best_rf_search.best_params_,
                     "BestModelPath": str(MODEL_PATH_RF), "Val_MAE": rf_val_mae, "Val_RMSE": rf_val_rmse, "Val_R2": rf_val_r2})
val_rows.append({"Model": "RandomForest", "Val_MAE": rf_val_mae, "Val_RMSE": rf_val_rmse, "Val_R2": rf_val_r2})

print(f"\nSelected RF branch: {'bootstrap=True' if best_rf_search is rf_search_boot else 'bootstrap=False'}")
print(f"Best CV params: {best_rf_search.best_params_}")
print(f"Validation MAE:  {rf_val_mae:.3f} | RMSE: {rf_val_rmse:.3f} | R2: {rf_val_r2:.3f}")
print(f"Saved model to:   {MODEL_PATH_RF.resolve()}")

# ---------- Save summary tables ----------
df_summary = pd.DataFrame(summary_rows).sort_values("Val_MAE")
df_val_rank = pd.DataFrame(val_rows).sort_values("Val_MAE")
df_summary.to_csv(OUT_SUMMARY, index=False)
df_val_rank.to_csv(OUT_VAL_RANK, index=False)

# Save best params (for paper appendix / future reuse)
with open(BEST_PARAMS_JSON, "w", encoding="utf-8") as f:
    json.dump(best_params_all, f, indent=2)

print_section("Validation Ranking (model selection; WITH shielding)")
print(f"Saved summary table: {OUT_SUMMARY.resolve()}")
print(f"Saved val ranking : {OUT_VAL_RANK.resolve()}")
print(f"Saved best params : {BEST_PARAMS_JSON.resolve()}")
display(prepare_display_table(df_val_rank))


### 11. Refined hyperparameter search with shielding

Purpose: Refine the top-performing shielding-aware models and archive their validation ranking.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 11. Refined hyperparameter search with shielding
# Purpose: Refine the top-performing shielding-aware models and archive their validation ranking.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import ast

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

# ---------- Reproducibility ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
CV_FOLDS = 5
SEARCH_NJOBS = 20

# Refinement controls
TOPK_SEEDS = 5                 # how many coarse "best" configs to seed refinement
N_ITER_PER_SEED = 40           # randomized iterations per seed (total per model = TOPK_SEEDS*N_ITER_PER_SEED)
CAT_TOPN = 2                   # keep top-N categorical values (based on coarse top rows) + best
K_NEIGHBORS = 2                # for ordered list params
INT_WINDOW = 3                 # refine int params as best±window
FLOAT_RATIO = 0.5              # refine float params as best*(1±ratio)
RANDOM_STATE_BASE = 1000       # to vary random_state across seeds while reproducible

# ---------- Paths ----------
PROJECT_ROOT = Path(".").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
TABLE_DIR = RESULTS_DIR / "tables"
MODEL_DIR = RESULTS_DIR / "models"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "features_dataset_with_shield.parquet"

# IMPORTANT: coarse CV outputs from Cell 12 (use the merged RF file you created)
CV_PATHS_COARSE = {
    "DecisionTree": TABLE_DIR / "cv_results_decisiontree_with_shield.csv",
    "RandomForest": TABLE_DIR / "cv_results_randomforest_with_shield.csv",
    "ExtraTrees": TABLE_DIR / "cv_results_extratrees_with_shield.csv",
}

# Outputs
OUT_REFINED_RANK = TABLE_DIR / "refined_validation_ranking_with_shield.csv"
OUT_REFINED_CV = {
    "DecisionTree": TABLE_DIR / "cv_results_refined_decisiontree_with_shield.csv",
    "RandomForest": TABLE_DIR / "cv_results_refined_randomforest_with_shield.csv",
    "ExtraTrees": TABLE_DIR / "cv_results_refined_extratrees_with_shield.csv",
}
OUT_REFINED_MODEL = {
    "DecisionTree": MODEL_DIR / "best_on_val_refined_DecisionTree_with_shield.pkl",
    "RandomForest": MODEL_DIR / "best_on_val_refined_RandomForest_with_shield.pkl",
    "ExtraTrees": MODEL_DIR / "best_on_val_refined_ExtraTrees_with_shield.pkl",
}

# ---------- Checks ----------
if not FEATURE_PATH.exists():
    raise FileNotFoundError(f"Feature dataset not found: {FEATURE_PATH}")
for m, p in CV_PATHS_COARSE.items():
    if not p.exists():
        raise FileNotFoundError(f"Coarse CV results for {m} not found: {p} (run Cell 12 + merge RF first)")

# ---------- Load dataset ----------
df_features = pd.read_parquet(FEATURE_PATH)
required_cols = {"SMILES", "shift_value"}
missing_cols = required_cols - set(df_features.columns)
if missing_cols:
    raise ValueError(f"Feature table missing required columns: {missing_cols}")

X = df_features.drop(columns=["SMILES", "shift_value"])
y = df_features["shift_value"].astype(float)

# ---------- Reproduce Train/Val/Test split (must match Cell 12) ----------
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(1/9), random_state=SEED)

print_section("Split Summary (reproducibility; TEST not used in refinement selection)")
print(f"Total samples : {len(X)}")
print(f"Train samples : {len(X_train)}")
print(f"Val samples   : {len(X_val)}")
print(f"Test samples  : {len(X_test)}")
print(f"Feature dim   : {X.shape[1]}")

# ---------- Metrics ----------
def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

# =========================
# Coarse spaces (for local narrowing)
# Must match Cell 12 ranges
# =========================
param_spaces_coarse = {
    "DecisionTree": {
        "criterion": ["squared_error", "friedman_mse", "absolute_error"],
        "splitter": ["best", "random"],
        "max_depth": [None, 5, 10, 20, 30, 50, 80, 120],
        "min_samples_split": [2, 5, 10, 20, 50, 100],
        "min_samples_leaf": [1, 2, 4, 8, 16, 32],
        "max_features": [None, "sqrt", "log2", 0.2, 0.4, 0.6, 0.8],
        "min_impurity_decrease": [0.0, 1e-7, 1e-6, 1e-5, 1e-4],
        "ccp_alpha": [0.0, 1e-7, 1e-6, 1e-5, 1e-4],
    },
    "RandomForest": {
        "n_estimators": [200, 400, 800, 1200, 1600],
        "criterion": ["squared_error", "absolute_error", "friedman_mse"],
        "max_depth": [None, 10, 20, 40, 60, 80, 120],
        "min_samples_split": [2, 5, 10, 20, 50],
        "min_samples_leaf": [1, 2, 4, 8, 16],
        "max_features": ["sqrt", "log2", None, 0.2, 0.4, 0.6, 0.8],
        "bootstrap": [True, False],
        "max_samples": [None, 0.5, 0.7, 0.9],
        "min_impurity_decrease": [0.0, 1e-7, 1e-6, 1e-5],
        "ccp_alpha": [0.0, 1e-7, 1e-6, 1e-5],
    },
    "ExtraTrees": {
        "n_estimators": [200, 400, 800, 1200, 1600],
        "criterion": ["squared_error", "absolute_error", "friedman_mse"],
        "max_depth": [None, 10, 20, 40, 60, 80, 120],
        "min_samples_split": [2, 5, 10, 20, 50],
        "min_samples_leaf": [1, 2, 4, 8, 16],
        "max_features": ["sqrt", "log2", None, 0.2, 0.4, 0.6, 0.8],
        "min_impurity_decrease": [0.0, 1e-7, 1e-6, 1e-5],
        "ccp_alpha": [0.0, 1e-7, 1e-6, 1e-5],
    },
}

# =========================
# Helpers for refined space
# =========================
def _is_categorical(v):
    return isinstance(v, (str, bool)) or v is None

def _dedup_keep_order(seq):
    out = []
    for x in seq:
        if x not in out:
            out.append(x)
    return out

def _refine_from_ordered_list(values, best_value, k=2):
    if best_value not in values:
        return [best_value]
    idx = values.index(best_value)
    lo = max(0, idx - k)
    hi = min(len(values), idx + k + 1)
    return _dedup_keep_order(values[lo:hi])

def _refine_int(best_value, window=3, min_value=1):
    lo = max(min_value, int(best_value) - window)
    hi = max(lo + 1, int(best_value) + window)
    return list(range(lo, hi + 1))

def _refine_float(best_value, ratio=0.5, n_points=7):
    best_value = float(best_value)
    if best_value == 0.0:
        return [0.0]
    lo = best_value * (1.0 - ratio)
    hi = best_value * (1.0 + ratio)
    if lo > 0 and hi > 0:
        return list(np.exp(np.linspace(np.log(lo), np.log(hi), n_points)))
    return list(np.linspace(lo, hi, n_points))

def _smart_refine_max_features(coarse_values, best_value, k=2):
    """
    max_features often mixes {None, 'sqrt', 'log2', floats}.
    Strategy:
    - Always keep best_value
    - Keep all non-numeric categorical options (None/'sqrt'/'log2') seen in coarse_values (small set)
    - For numeric best_value -> keep numeric neighbors
    """
    cats = [v for v in coarse_values if isinstance(v, (str, type(None)))]
    nums = [v for v in coarse_values if isinstance(v, (int, float)) and not isinstance(v, bool)]

    out = []
    # keep best
    out.append(best_value)

    # keep categorical set (small)
    for v in cats:
        if v not in out:
            out.append(v)

    # if best is numeric, keep numeric neighbors; if best is categorical, keep a few numeric midpoints
    if isinstance(best_value, (int, float)) and not isinstance(best_value, bool):
        nums_sorted = sorted(nums)
        out += _refine_from_ordered_list(nums_sorted, float(best_value), k=k)
    else:
        # include a small subset of numeric defaults
        for v in [0.2, 0.4, 0.6]:
            if v in nums and v not in out:
                out.append(v)

    return _dedup_keep_order(out)

def top_categorical_candidates_from_coarse(cv_df, param_name, topk=20, keep_n=2):
    """
    Look at top-k rows (by rank_test_score) and collect most frequent values for a categorical param.
    Return up to keep_n values (ordered by frequency then first occurrence).
    """
    if "rank_test_score" not in cv_df.columns or "params" not in cv_df.columns:
        return None
    top = cv_df.sort_values("rank_test_score").head(topk)
    vals = []
    for s in top["params"].tolist():
        d = ast.literal_eval(s)
        if param_name in d:
            vals.append(d[param_name])
    if not vals:
        return None
    # frequency
    counts = {}
    first_idx = {}
    for i, v in enumerate(vals):
        counts[v] = counts.get(v, 0) + 1
        if v not in first_idx:
            first_idx[v] = i
    ranked = sorted(counts.keys(), key=lambda v: (-counts[v], first_idx[v]))
    return ranked[:keep_n]

def build_refined_space(model_name, best_params, coarse_space, cv_df_for_cats=None,
                        k_neighbors=2, int_window=3, float_ratio=0.5, cat_topn=2):
    """
    Improvements vs original:
    - Use coarse top rows to allow a SMALL set of categorical options (not fixed to single value).
    - Special handling for max_features mixed types.
    - RF bootstrap branch consistent: if best bootstrap is False, we drop max_samples.
    """
    refined = {}

    for p, best_v in best_params.items():
        # RF constraint: max_samples only when bootstrap=True
        if model_name == "RandomForest" and p == "max_samples":
            if best_params.get("bootstrap", False) is False:
                continue

        if p in coarse_space and isinstance(coarse_space[p], list):
            # max_features special
            if p == "max_features":
                refined[p] = _smart_refine_max_features(coarse_space[p], best_v, k=k_neighbors)
                continue

            # categorical: keep best + a couple of popular alternatives from coarse top rows
            if _is_categorical(best_v):
                cand = [best_v]
                if cv_df_for_cats is not None:
                    extra = top_categorical_candidates_from_coarse(cv_df_for_cats, p, topk=30, keep_n=cat_topn)
                    if extra:
                        for v in extra:
                            if v not in cand:
                                cand.append(v)
                refined[p] = cand
            else:
                refined[p] = _refine_from_ordered_list(coarse_space[p], best_v, k=k_neighbors)
            continue

        # fallback by type
        if _is_categorical(best_v):
            refined[p] = [best_v]
        elif isinstance(best_v, int):
            refined[p] = _refine_int(best_v, window=int_window, min_value=1)
        elif isinstance(best_v, float):
            refined[p] = _refine_float(best_v, ratio=float_ratio, n_points=7)
        else:
            refined[p] = [best_v]

    # Final RF safety: enforce bootstrap branch consistency
    if model_name == "RandomForest":
        boot = best_params.get("bootstrap", None)
        if boot is not None:
            refined["bootstrap"] = [boot]  # FIX branch
            if boot is False:
                refined.pop("max_samples", None)

    return refined

# =========================
# Model factories
# =========================
def make_estimator(model_name):
    if model_name == "DecisionTree":
        return DecisionTreeRegressor(random_state=SEED)
    if model_name == "RandomForest":
        return RandomForestRegressor(random_state=SEED, n_jobs=SEARCH_NJOBS)
    if model_name == "ExtraTrees":
        return ExtraTreesRegressor(random_state=SEED, n_jobs=SEARCH_NJOBS)
    raise ValueError(f"Unknown model: {model_name}")

# =========================
# Main refinement (Top-K seeds)
# =========================
refined_rank_rows = []

for model_name in ["DecisionTree", "RandomForest", "ExtraTrees"]:
    print(f"\n==============================")
    print(f"Refinement: {model_name} (Top-{TOPK_SEEDS} seeds)")
    print(f"==============================")

    cv_df = pd.read_csv(CV_PATHS_COARSE[model_name])

    if ("rank_test_score" not in cv_df.columns) or ("params" not in cv_df.columns):
        raise ValueError(f"{CV_PATHS_COARSE[model_name]} missing required columns: rank_test_score / params")

    # ---- Take Top-K coarse seeds (by rank_test_score) ----
    cv_top = cv_df.sort_values("rank_test_score").head(TOPK_SEEDS).copy()
    seed_params_list = [ast.literal_eval(s) for s in cv_top["params"].tolist()]

    all_refined_results = []
    best_refined_estimator = None
    best_refined_cv_score = -np.inf  # higher neg-MAE is better

    # We'll collect all cv_results_ across seeds (tagged) for auditing/paper
    cv_results_all_seeds = []

    for seed_idx, best_params in enumerate(seed_params_list, start=1):
        print(f"\n--- Seed {seed_idx}/{TOPK_SEEDS}: coarse-best params ---")
        print(prepare_display_table(pd.DataFrame([best_params])))

        # Build refined space around this seed
        refined_space = build_refined_space(
            model_name=model_name,
            best_params=best_params,
            coarse_space=param_spaces_coarse[model_name],
            cv_df_for_cats=cv_df,
            k_neighbors=K_NEIGHBORS,
            int_window=INT_WINDOW,
            float_ratio=FLOAT_RATIO,
            cat_topn=CAT_TOPN,
        )

        # Quick summary
        print("Refined space keys:", list(refined_space.keys()))
        print("Refined space sizes:", {k: len(v) for k, v in refined_space.items()})

        estimator = make_estimator(model_name)

        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=refined_space,
            n_iter=N_ITER_PER_SEED,
            cv=CV_FOLDS,
            scoring="neg_mean_absolute_error",
            n_jobs=-1,
            random_state=RANDOM_STATE_BASE + seed_idx,  # vary per seed but reproducible
            verbose=1,
            return_train_score=True,
        )
        search.fit(X_train, y_train)

        # Track best across seeds by CV score (TRAIN CV)
        if search.best_score_ > best_refined_cv_score:
            best_refined_cv_score = search.best_score_
            best_refined_estimator = search.best_estimator_

        # Store detailed results
        df_seed = pd.DataFrame(search.cv_results_)
        df_seed["SeedIndex"] = seed_idx
        df_seed["CoarseSeedParams"] = str(best_params)
        cv_results_all_seeds.append(df_seed)

        all_refined_results.append({
            "SeedIndex": seed_idx,
            "BestScore_CVTrain_negMAE": search.best_score_,
            "BestParams_CVTrain": search.best_params_,
        })

        print(f"Seed {seed_idx} best CV neg-MAE: {search.best_score_:.6f}")
        print(f"Seed {seed_idx} best params: {search.best_params_}")

    # ---- Save merged refined CV results (all seeds) ----
    df_cv_refined_all = pd.concat(cv_results_all_seeds, ignore_index=True)
    df_cv_refined_all.to_csv(OUT_REFINED_CV[model_name], index=False)
    print(f"\nSaved refined CV results (ALL seeds) to:\n  {OUT_REFINED_CV[model_name].resolve()}")

    # ---- Evaluate the globally best refined estimator on VAL ----
    y_val_pred = best_refined_estimator.predict(X_val)
    val_mae, val_rmse, val_r2 = evaluate_regression(y_val, y_val_pred)

    # Save refined model
    joblib.dump(best_refined_estimator, OUT_REFINED_MODEL[model_name])
    print(f"Saved refined best model to:\n  {OUT_REFINED_MODEL[model_name].resolve()}")

    # Record for ranking
    refined_rank_rows.append({
        "Model": model_name,
        "Val_MAE": val_mae,
        "Val_RMSE": val_rmse,
        "Val_R2": val_r2,
        "BestRefined_CVTrain_negMAE": float(best_refined_cv_score),
        "RefinedModelPath": str(OUT_REFINED_MODEL[model_name]),
        "RefinedCVPath": str(OUT_REFINED_CV[model_name]),
        "TopKSeedsUsed": TOPK_SEEDS,
        "NIterPerSeed": N_ITER_PER_SEED,
        "TotalItersApprox": TOPK_SEEDS * N_ITER_PER_SEED,
    })

    print("\nValidation performance (Best Refined across seeds):")
    print(f"Val MAE : {val_mae:.3f}")
    print(f"Val RMSE: {val_rmse:.3f}")
    print(f"Val R2  : {val_r2:.3f}")

# ---- Save refined ranking ----
df_refined_rank = pd.DataFrame(refined_rank_rows).sort_values("Val_MAE")
df_refined_rank.to_csv(OUT_REFINED_RANK, index=False)

print_section("Refined Validation Ranking (ALL models; Top-K seeds refinement)")
print(f"Saved to: {OUT_REFINED_RANK.resolve()}")
display(prepare_display_table(df_refined_rank))


### 12. Train, validation, and test parity analysis

Purpose: Visualize the agreement between observed and predicted shifts across all data splits for the selected model.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 12. Train, validation, and test parity analysis
# Purpose: Visualize the agreement between observed and predicted shifts across all data splits for the selected model.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42

PROJECT_ROOT = Path('.').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
MODEL_DIR = RESULTS_DIR / 'models'
FIG_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'features_dataset.parquet'
if not FEATURE_PATH.exists():
    raise FileNotFoundError(f'Feature dataset not found: {FEATURE_PATH}')

MODEL_PATH = MODEL_DIR / 'best_on_val_refined_ExtraTrees.pkl'
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model not found: {MODEL_PATH}')

OUT_FIG = FIG_DIR / 'best_on_val_refined_ExtraTrees.png'

apply_plot_style()

df_features = pd.read_parquet(FEATURE_PATH)

X = df_features.drop(columns=['SMILES', 'shift_value'])
y = df_features['shift_value'].astype(float)

print_section('Dataset loaded')
print(f'Samples      : {len(df_features)}')
print(f'Feature dim  : {X.shape[1]}')
print(f'Feature file : {FEATURE_PATH.resolve()}')
print(f'Model file   : {MODEL_PATH.resolve()}')

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=(1 / 9), random_state=SEED
)

print_section('Split Summary')
print(f'Train samples : {len(X_train)}')
print(f'Val samples   : {len(X_val)}')
print(f'Test samples  : {len(X_test)}')

def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

def parity_plot(ax, y_true, y_pred, title):
    ax.scatter(y_true, y_pred, s=18, alpha=0.75, edgecolor='none')

    y_min, y_max = compute_plot_limits(y_true, y_pred, pad_ratio=0.0)
    add_identity_line(ax, y_min, y_max)

    ax.set_title(title)
    ax.set_xlabel('True shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    ax.set_xlim(y_min, y_max)
    ax.set_ylim(y_min, y_max)
    style_axes(ax, equal_aspect=True)

loaded = joblib.load(MODEL_PATH)
if isinstance(loaded, dict):
    for _k in ['model', 'best_model', 'best_estimator', 'best_estimator_', 'estimator']:
        if _k in loaded:
            model = loaded[_k]
            break
    else:
        raise TypeError(f'Loaded object is a dict but no known model key found. Available keys: {list(loaded.keys())}')
else:
    model = loaded
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

train_mae, train_rmse, train_r2 = regression_metrics(y_train, y_pred_train)
val_mae, val_rmse, val_r2 = regression_metrics(y_val, y_pred_val)
test_mae, test_rmse, test_r2 = regression_metrics(y_test, y_pred_test)

print_section('Performance')
print(metric_row('Train', train_mae, train_rmse, train_r2, label_width=10))
print(metric_row('Validation', val_mae, val_rmse, val_r2, label_width=12))
print(metric_row('Test', test_mae, test_rmse, test_r2, label_width=12))

fig, axes = create_figure('triptych', nrows=1, ncols=3)
ax1, ax2, ax3 = axes

parity_plot(ax1, y_train.values, y_pred_train, 'Train')
annotate_panel_text(ax1, metric_text(train_mae, train_rmse, train_r2, decimals=2))

parity_plot(ax2, y_val.values, y_pred_val, 'Validation')
annotate_panel_text(ax2, metric_text(val_mae, val_rmse, val_r2, decimals=2))

parity_plot(ax3, y_test.values, y_pred_test, 'Test')
annotate_panel_text(ax3, metric_text(test_mae, test_rmse, test_r2, decimals=2))

fig.suptitle(f'Parity plots with shielding: {MODEL_PATH.stem}', fontsize=13)
fig.tight_layout()

saved_paths = save_figure(fig, OUT_FIG)
plt.show()

report_saved_paths(saved_paths, 'Saved parity-plot files')


### 13. Compare ExtraTrees Models With and Without Shielding

Purpose: Compare the structure-only and shielding-aware ExtraTrees models on the held-out test split using the saved refined models.


In [ ]:
# -----------------------------------------------------------------------------
# Compare ExtraTrees models with and without shielding
# Purpose: Load the saved refined ExtraTrees models, evaluate them on the aligned test split, and export the side-by-side comparison plot.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42

FEATURE_NO_SIGMA = Path('data/processed/features_dataset.parquet')
FEATURE_WITH_SIGMA = Path('data/processed/features_dataset_with_shield.parquet')

MODEL_NO_SIGMA = Path('results/models/best_on_val_refined_ExtraTrees.pkl')
MODEL_WITH_SIGMA = Path('results/models/best_on_val_refined_ExtraTrees_with_shield.pkl')

OUT_PNG = Path('results/figures/Figure4_ExtraTrees_Test_AB.png')
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)

apply_plot_style()

def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

def load_xy_predict(feature_path, model_path):
    df = pd.read_parquet(feature_path)
    X = df.drop(columns=['SMILES', 'shift_value'])
    y = df['shift_value'].astype(float)

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.10, random_state=SEED
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=(1 / 9), random_state=SEED
    )
    model = joblib.load(model_path)
    y_pred_test = model.predict(X_test)

    mae, rmse, r2 = regression_metrics(y_test, y_pred_test)
    return y_test.values, y_pred_test, (mae, rmse, r2)

def parity(ax, y_true, y_pred, title, metrics, vmin, vmax):
    mae, rmse, r2 = metrics
    ax.scatter(y_true, y_pred, s=18, alpha=0.80, edgecolor='none')
    add_identity_line(ax, vmin, vmax)

    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_title(title)
    ax.set_xlabel('True shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    style_axes(ax, equal_aspect=True)

    annotate_panel_text(
        ax,
        f'MAE={mae:.2f} ppm\nRMSE={rmse:.2f} ppm\nR$^2$={r2:.3f}',
    )

y_true_A, y_pred_A, mA = load_xy_predict(FEATURE_NO_SIGMA, MODEL_NO_SIGMA)
y_true_B, y_pred_B, mB = load_xy_predict(FEATURE_WITH_SIGMA, MODEL_WITH_SIGMA)

vmin, vmax = compute_plot_limits(y_true_A, y_pred_A, y_true_B, y_pred_B)

fig, axes = create_figure('comparison', nrows=1, ncols=2, constrained_layout=True)

parity(axes[0], y_true_A, y_pred_A, 'ExtraTrees (structural only) - Test', mA, vmin, vmax)
parity(axes[1], y_true_B, y_pred_B, 'ExtraTrees (structural + shielding) - Test', mB, vmin, vmax)

saved_paths = save_figure(fig, OUT_PNG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')
